# **K-NN Histogram Classification via Semantic Clustering**

In this notebook, we transition from analyzing Out-of-Distribution (OOD) subjects to building a quantifiable classification pipeline. Building upon our previous experiments segmenting and clustering semantic features, we will now attempt to classify images based entirely on their feature distributions (Bag of Visual Words histograms).

To establish a mathematical baseline and validate our architecture, we will begin our experiments on a fundamental computer vision dataset: the **MNIST handwritten digits**.

Our methodology involves:
1. Extracting deep spatial features and grouping them into semantic clusters.
2. Constructing frequency histograms that represent the distinct "semantic fingerprint" of each image.
3. Utilizing a **K-Nearest Neighbors (K-NN)** classifier—equipped with a **Chi-Square distance metric**—to classify the digits based on their histogram similarities.

Furthermore, we will rigorously experiment with the vocabulary size (the number of K-Means clusters, $K$). We will test a spectrum of spatial resolutions, ranging from low-dimensional conceptual spaces ($K=4$) to highly fractured, micro-textural spaces ($K \ge 1000$).

Successfully validating this pipeline on MNIST will serve as the foundational proof-of-concept required to transition into complex **Hierarchical Classification** in the near future.

## **0. Model Preparation**

### **Imports**

In [3]:
import torch
import numpy as np
import torchvision
import matplotlib.pyplot as plt
import kagglehub
from transformers import AutoImageProcessor, AutoModel

### **Model Preparation**

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [5]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [6]:
model_name = "facebook/dinov3-vit7b16-pretrain-lvd1689m"

print(f"Loading flagship model: {model_name}")
print(f"Targeting Architecture: 40 Layers, 4096 Embedding Dim")

if 'dtype' not in locals():
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32

try:
    #Load Model with Memory Optimization and Eager Attention
    # We strictly need eager attention to capture the attention weights
    model = AutoModel.from_pretrained(
        model_name,
        torch_dtype=dtype,
        low_cpu_mem_usage=True,
        attn_implementation="eager"
    ).to(device).eval()

    #Load Processor
    processor = AutoImageProcessor.from_pretrained(model_name)

    num_registers = getattr(model.config, "num_register_tokens", 4)
    print(f"Model loaded. Number of register tokens identified: {num_registers}")

    transform = lambda img: processor(images=img, return_tensors="pt")['pixel_values'].squeeze()

    #Verify Layer Count
    if hasattr(model, "vision_model"):
        num_layers = len(model.vision_model.encoder.layers)
    else:
        #Fallback for flat architectures
        num_layers = len(model.layer)

    print(f"Success! Model loaded with {num_layers} layers.")

except Exception as e:
    print(f"Error loading model: {e}")
    print("Tip: If you run out of memory (OOM), try 'dinov3-vitl16' (Large, 24 layers) instead.")

Loading flagship model: facebook/dinov3-vit7b16-pretrain-lvd1689m
Targeting Architecture: 40 Layers, 4096 Embedding Dim


config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

KeyboardInterrupt: 

## **1. Dataset Prep**

### **1.1 Imports**

In [ ]:
from torchvision import transforms
import matplotlib.pyplot as plt

### **1.2 Downloading the MNIST Numbers**

In [ ]:
#Define a basic transform (Convert to PIL Image so it matches our DINOv3 pipeline)
# DINO expects RGB images, but MNIST is grayscale (1 channel).
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3), # Convert 1-channel to 3-channel RGB
    transforms.Resize((224, 224)),               # Resize to a standard DINOv3 resolution
])

In [ ]:
mnist_train_full = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
mnist_test_full = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)